# Lab 7 Tasks

This notebook contains solutions for Task01, Task02, and Task03 as requested.

## Task01: Warehouse robot diagonal path CSP

1. Variables
- `x[k]`, `y[k]` for each step `k` in path index (0..max_steps-1)
- `step_used[k]` (0/1) to allow shorter path than max steps optionally

2. Domains
- `x[k]`, `y[k]` integer in [0, 4] (for 5x5 grid)
- `step_used[k]` in {0,1}

3. Constraints
- Start: `(x[0], y[0]) == (1,1)` (converted to 0-based `(0,0)`)
- End: `(x[last], y[last]) == (4,4)` (0-based `(3,3)`)
- Travel: each move diagonal => `|x[k+1]-x[k]| == 1` and `|y[k+1]-y[k]| == 1`
- Obstacles: positions cannot be an obstacle cell
- Path termination: once goal reached, subsequent steps can stay at goal (or `step_used` to ignore)
- Objective: minimize number of steps (equivalently shortest Euclidean path with `sqrt(2)` per step.)

In [1]:
# Task01 OR-Tools CSP and shortest diagonal path
try:
    from ortools.sat.python import cp_model
except ImportError:
    !pip install ortools -q
    from ortools.sat.python import cp_model

N = 5
start = (0, 0)  # (1,1) -> 0-based
goal = (3, 3)   # (4,4) -> 0-based
obstacles = {(1, 2), (2, 1)}
start_id = start[0] * N + start[1]
goal_id = goal[0] * N + goal[1]

# All diagonal neighbors allowed set per node
diag_moves = {}
for i in range(N):
    for j in range(N):
        if (i, j) in obstacles:
            continue
        nid = i * N + j
        nei = []
        for di, dj in [(-1, -1), (-1, 1), (1, -1), (1, 1)]:
            ni = i + di
            nj = j + dj
            if 0 <= ni < N and 0 <= nj < N and (ni, nj) not in obstacles:
                nei.append(ni * N + nj)
        diag_moves[nid] = nei

# Try increasing path length until feasible (shortest path)
solution_path = None
for path_len in range(4, 12):  # lower bound for this map is 3 diag steps, upper bound 11
    model = cp_model.CpModel()
    nodes = [model.NewIntVar(0, N * N - 1, f'node_{k}') for k in range(path_len)]

    model.Add(nodes[0] == start_id)
    model.Add(nodes[path_len - 1] == goal_id)

    # obstacle avoidance
    for k in range(path_len):
        for ox, oy in obstacles:
            model.Add(nodes[k] != ox * N + oy)

    # adjacency constraints per step
    for k in range(path_len - 1):
        allowed = []
        for u, neighbors in diag_moves.items():
            for v in neighbors:
                allowed.append((u, v))
        model.AddAllowedAssignments([nodes[k], nodes[k + 1]], allowed)

    # prevent immediate cycles is optional but helps efficiency
    model.AddAllDifferent(nodes)

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 5
    solver.parameters.num_search_workers = 8
    status = solver.Solve(model)

    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        solution_path = [solver.Value(nodes[k]) for k in range(path_len)]
        break

if solution_path is None:
    print('Task01 no solution found')
else:
    coord_path = [(p // N, p % N) for p in solution_path]
    print('Task01 shortest diagonal path:', coord_path)
    print('Steps:', len(coord_path) - 1, 'cost approx', round((len(coord_path) - 1) * 2**0.5, 3))

Task01 shortest diagonal path: [(0, 0), (1, 1), (2, 2), (3, 3)]
Steps: 3 cost approx 4.243


## Task02: Island landmass perimeter as CSP

1. Variables
- `cell[i][j]` binary variable (0 water, 1 land) for each grid cell.

2. Normal constraints
- `cell[i][j]` in {0,1}
- Known map can be pinned via equality constraints (for this problem, input is fixed).

3. Boundary edges/perimeter
- For each land cell, perimeter contributions come from: top, bottom, left, right where adjacent cell is water or out-of-bounds.
- `edge[i][j][d]` binary var (1 if boundary edge exists in direction `d`).
- Sum of all `edge` variables = total perimeter.

In [2]:
# Task02 OR-Tools model for landmass perimeter
try:
    from ortools.sat.python import cp_model
except ImportError:
    !pip install ortools -q
    from ortools.sat.python import cp_model

map_grid = [
    [0,1,1,0,0],
    [1,1,1,0,0],
    [0,1,0,1,1],
    [0,0,0,1,0],
    [0,0,0,0,0]
]  # example 5x5

N = len(map_grid)
model = cp_model.CpModel()
cell = [[model.NewBoolVar(f'cell_{i}_{j}') for j in range(N)] for i in range(N)]

# Fix to known map input
for i in range(N):
    for j in range(N):
        if map_grid[i][j] == 1:
            model.Add(cell[i][j] == 1)
        else:
            model.Add(cell[i][j] == 0)

# perimeter edges
perim_terms = []
for i in range(N):
    for j in range(N):
        # if cell is land, check 4 neighbors
        is_land = cell[i][j]
        # top
        if i == 0:
            perim = model.NewIntVar(0, 1, f'p_{i}_{j}_top')
            model.Add(perim == is_land)
            perim_terms.append(perim)
        else:
            boundary = model.NewIntVar(0, 1, f'p_{i}_{j}_top')
            # boundary is 1 when land and top neighbor is water
            # top is conditional by linearization: boundary >= land - neighbor and boundary <= land, boundary <= 1 - neighbor
            model.Add(boundary >= is_land - cell[i-1][j])
            model.Add(boundary <= is_land)
            model.Add(boundary <= 1 - cell[i-1][j])
            perim_terms.append(boundary)

        # bottom
        if i == N-1:
            perim = model.NewIntVar(0, 1, f'p_{i}_{j}_bottom')
            model.Add(perim == is_land)
            perim_terms.append(perim)
        else:
            boundary = model.NewIntVar(0, 1, f'p_{i}_{j}_bottom')
            model.Add(boundary >= is_land - cell[i+1][j])
            model.Add(boundary <= is_land)
            model.Add(boundary <= 1 - cell[i+1][j])
            perim_terms.append(boundary)

        # left
        if j == 0:
            perim = model.NewIntVar(0, 1, f'p_{i}_{j}_left')
            model.Add(perim == is_land)
            perim_terms.append(perim)
        else:
            boundary = model.NewIntVar(0, 1, f'p_{i}_{j}_left')
            model.Add(boundary >= is_land - cell[i][j-1])
            model.Add(boundary <= is_land)
            model.Add(boundary <= 1 - cell[i][j-1])
            perim_terms.append(boundary)

        # right
        if j == N-1:
            perim = model.NewIntVar(0, 1, f'p_{i}_{j}_right')
            model.Add(perim == is_land)
            perim_terms.append(perim)
        else:
            boundary = model.NewIntVar(0, 1, f'p_{i}_{j}_right')
            model.Add(boundary >= is_land - cell[i][j+1])
            model.Add(boundary <= is_land)
            model.Add(boundary <= 1 - cell[i][j+1])
            perim_terms.append(boundary)

perimeter = model.NewIntVar(0, 4*N*N, 'perimeter')
model.Add(perimeter == sum(perim_terms))
model.Minimize(perimeter)  # Not needed but triggers solver

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 5
status = solver.Solve(model)

if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print('Task02 perimeter:', solver.Value(perimeter))
else:
    print('Task02 no solution available')


Task02 perimeter: 20


## Task03: Salesperson TSP for 10 cities

1. Variables
- `next_city[i]` as successor of city `i` in route.
- `visit_order[i]` (optional) ordering of visits.

2. Domains
- `next_city[i]` in [0,9], and all different.
- Start and end can be enforced via route fixed start.

3. Constraints
- single cycle through all 10 cities
- no sub-cycles (handled by OR-Tools routing solver)
- distance matrix cost minimization.

Uses OR-Tools routing manager and solver.

In [3]:
# Task03 OR-Tools Routing TSP for 10 cities
try:
    from ortools.constraint_solver import pywrapcp
    from ortools.constraint_solver import routing_enums_pb2
except ImportError:
    !pip install ortools -q
    from ortools.constraint_solver import pywrapcp
    from ortools.constraint_solver import routing_enums_pb2

# Example city coordinates
towns = [
    (0,0), (1,5), (2,3), (5,2), (6,6),
    (7,1), (8,5), (3,8), (9,0), (4,4)
]
num_cities = len(towns)

def manhattan_dist(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

distance_matrix = [[manhattan_dist(a, b) for b in towns] for a in towns]

manager = pywrapcp.RoutingIndexManager(num_cities, 1, 0)
routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)
search_parameters.time_limit.seconds = 10

solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    plan = []
    route_distance = 0
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        plan.append(node)
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(previous_index, index, 0)
    plan.append(manager.IndexToNode(index))

    print('Task03 optimal cycle path:', plan)
    print('Total distance:', route_distance)
else:
    print('Task03 no solution found')

Task03 optimal cycle path: [0, 2, 9, 3, 5, 8, 6, 4, 7, 1, 0]
Total distance: 42
